In [1]:
import pandas as pd 

df = pd.read_csv('clean_df.csv')
print(df.columns)

Index(['Country', 'ASN', 'TTL', 'IP', 'Domain', 'State', 'Registrant_Name',
       'Domain_Name', 'subdomain', 'Organization', 'len', 'longest_word',
       'obfuscate_at_sign', 'entropy', 'Domain_Age', 'tld', 'Emails',
       'numeric_percentage', 'char_distribution', 'Registrar', 'sld',
       'Name_Server_Count', 'label'],
      dtype='object')


## IP to IP Subnet

In [2]:
df['IP_Subnet'] = df['IP'].astype(str).apply(lambda x: '.'.join(x.split('.')[:2]))
df['IP_Subnet'] = df['IP_Subnet'].astype('category')
df.drop(columns=['IP'], inplace=True)

## Handle Missing Values and change data type

In [3]:
categorical_cols = ["Country", "State", "Registrar", "Organization", "tld", "sld"]

for col in categorical_cols:
    df[col] = df[col].fillna('unknown')
    df[col] = df[col].astype('category')

In [4]:
num_cols = ['ASN', 'TTL', 'len', 'entropy', 'numeric_percentage', 'Name_Server_Count']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
if 'Domain_Age' in df.columns:
    df['Domain_Age'] = df['Domain_Age'].astype(str).str.extract(r'(\d+)').astype(float).fillna(-1)

In [6]:
object_cols = df.select_dtypes(include=['object']).columns.tolist()
object_cols

['Domain',
 'Registrant_Name',
 'Domain_Name',
 'longest_word',
 'Emails',
 'char_distribution']

In [7]:
for col in object_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna('unknown')

In [8]:
print(df.dtypes)

Country               category
ASN                    float64
TTL                      int64
Domain                  object
State                 category
Registrant_Name         object
Domain_Name             object
subdomain              float64
Organization          category
len                    float64
longest_word            object
obfuscate_at_sign        int64
entropy                float64
Domain_Age             float64
tld                   category
Emails                  object
numeric_percentage     float64
char_distribution       object
Registrar             category
sld                   category
Name_Server_Count      float64
label                    int64
IP_Subnet             category
dtype: object


## Handle list columns

In [9]:
df['char_distribution']

0         defaultdict(<class 'int'>, {'e': 1, 'o': 2, 'g...
1         defaultdict(<class 'int'>, {'e': 1, 'o': 2, 'g...
2         defaultdict(<class 'int'>, {'e': 1, 'o': 2, 'w...
3         defaultdict(<class 'int'>, {'e': 1, 'o': 2, 'w...
4         defaultdict(<class 'int'>, {'e': 1, 'c': 1, 'o...
                                ...                        
485357    defaultdict(<class 'int'>, {'-': 2, 'b': 1, 'n...
485358    defaultdict(<class 'int'>, {'b': 1, 'c': 1, 'e...
485359    defaultdict(<class 'int'>, {'-': 2, 'i': 1, 't...
485360    defaultdict(<class 'int'>, {'-': 2, 'i': 1, 'x...
485361    defaultdict(<class 'int'>, {'p': 1, 'k': 1, 'i...
Name: char_distribution, Length: 485362, dtype: object

In [10]:
import re
vowel_patterns = {v: re.compile(fr"'{v}': (\d+)", re.IGNORECASE) for v in 'aeiou'}
def count_vowels_optimized(dist_str):
    if pd.isna(dist_str) or not isinstance(dist_str, str) or dist_str == 'nan':
        return 0
    
    total_vowels = 0
    for v, pattern in vowel_patterns.items():
        match = pattern.search(dist_str)
        if match:
            total_vowels += int(match.group(1))
            
    return total_vowels

df['vowel_count'] = df['char_distribution'].apply(count_vowels_optimized)
df['vowel_ratio'] = df['vowel_count'] / (df['len'] + 1)
df = df.drop(columns=['char_distribution'])

In [11]:
print(df.columns)

print("columns count:", len(df.columns))

Index(['Country', 'ASN', 'TTL', 'Domain', 'State', 'Registrant_Name',
       'Domain_Name', 'subdomain', 'Organization', 'len', 'longest_word',
       'obfuscate_at_sign', 'entropy', 'Domain_Age', 'tld', 'Emails',
       'numeric_percentage', 'Registrar', 'sld', 'Name_Server_Count', 'label',
       'IP_Subnet', 'vowel_count', 'vowel_ratio'],
      dtype='object')
columns count: 24


In [12]:
print(df.dtypes)

Country               category
ASN                    float64
TTL                      int64
Domain                  object
State                 category
Registrant_Name         object
Domain_Name             object
subdomain              float64
Organization          category
len                    float64
longest_word            object
obfuscate_at_sign        int64
entropy                float64
Domain_Age             float64
tld                   category
Emails                  object
numeric_percentage     float64
Registrar             category
sld                   category
Name_Server_Count      float64
label                    int64
IP_Subnet             category
vowel_count              int64
vowel_ratio            float64
dtype: object


## Change Order

In [13]:
print(df.dtypes)

Country               category
ASN                    float64
TTL                      int64
Domain                  object
State                 category
Registrant_Name         object
Domain_Name             object
subdomain              float64
Organization          category
len                    float64
longest_word            object
obfuscate_at_sign        int64
entropy                float64
Domain_Age             float64
tld                   category
Emails                  object
numeric_percentage     float64
Registrar             category
sld                   category
Name_Server_Count      float64
label                    int64
IP_Subnet             category
vowel_count              int64
vowel_ratio            float64
dtype: object


In [14]:
new_order = [
    # --- Numerical Features ---
    'ASN', 'TTL', 'subdomain', 'len', 'entropy', 'Domain_Age', 
    'numeric_percentage', 'Name_Server_Count', 'obfuscate_at_sign',
    'vowel_count', 'vowel_ratio',
    
    # --- Categorical Features ---
    'Country', 'State', 'Organization', 'tld', 'Registrar', 'sld', 'IP_Subnet',
    
    # --- Text/Sequence Features ---
    'Domain', 'Domain_Name', 'Registrant_Name', 'longest_word', 'Emails',
     # --- Target ---
    'label'
    
]
print("Before reordering:", len(df.columns))
existing_cols = [col for col in new_order if col in df.columns]
df = df[existing_cols]
print("After reordering:", len(df.columns)) 

Before reordering: 24
After reordering: 24


## Domain

In [15]:
df["Domain"].head()

0        b'google.com.'
1        b'google.com.'
2    b'www.google.com.'
3    b'www.google.com.'
4      b'facebook.com.'
Name: Domain, dtype: object

In [16]:
def clean_domain(domain):
    d = str(domain).strip()
    if d.startswith("b'") and d.endswith("'"):
        d = d[2:-1]
    else:
        print(f"no b'' from domain: {d}")
        return None
    if d.endswith("."):
        d = d[:-1]
    else:
        print(f"no trailing dot from domain: {d}")
        return None
    return d.lower()

df['Domain'] = df['Domain'].apply(clean_domain)

no b'' from domain: 397220
no b'' from domain: 397220
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81
no b'' from domain: 205.144.171.81


In [17]:
initial_count = len(df)
df = df.dropna(subset=['Domain']).reset_index(drop=True)
dropped_count = initial_count - len(df)
print(f"Dropped {dropped_count} rows due to invalid 'Domain' entries.")
print(len(df))

Dropped 12 rows due to invalid 'Domain' entries.
485350


In [18]:
df.to_csv("df_final.csv", index=False)

In [20]:
df.describe()

,ASN,TTL,subdomain,len,entropy,Domain_Age,numeric_percentage,Name_Server_Count,obfuscate_at_sign,vowel_count,vowel_ratio,label
count,378370.000000,485350.000000,485350.000000,485349.000000,485350.000000,485350.000000,485350.000000,428954.000000,485350.0,485350.000000,485349.000000,485350.000000
mean,36225.632553,4560.368684,0.326367,12.126509,2.667541,3409.090316,0.938280,3.435716,0.0,3.723985,0.269704,0.054859
std,53453.050636,6568.106652,0.468888,5.072213,0.542937,3250.761265,4.445528,4.376600,0.0,2.180357,0.106008,0.352002
min,3.000000,0.000000,-1.000000,1.000000,-0.000000,-1.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
25%,13335.000000,299.000000,0.000000,8.000000,2.373267,0.000000,0.000000,2.000000,0.0,2.000000,0.214286,0.000000
50%,19871.000000,898.000000,0.000000,11.000000,2.742338,2781.000000,0.000000,2.000000,0.0,3.000000,0.285714,0.000000
75%,40034.000000,3599.000000,1.000000,15.000000,3.062772,6439.000000,0.000000,4.000000,0.0,5.000000,0.333333,0.000000
max,398246.000000,46844.000000,1.000000,142.000000,4.766781,12830.000000,72.727273,86.000000,0.0,32.000000,0.833333,3.000000


In [23]:
len(df.columns)

24

In [22]:
df["label"].value_counts()

label
0    471893
1      4605
2      4535
3      4317
Name: count, dtype: int64